# 04 — Backtest Model Portfolio

This notebook converts the final trend signal and selected volatility forecasts into monthly portfolio returns. Weights are formed at month-end and applied to the following month's returns.


In [1180]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

pd.options.display.float_format = "{:,.4f}".format

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
TREND_DIR = RESULTS_DIR / "trend_signal"
VOL_DIR = RESULTS_DIR / "volatility_forecast"
OUT_DIR = RESULTS_DIR / "backtest_model_portfolio"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BALANCED_BENCHMARK_WEIGHTS = {
    "SPY": 0.25,
    "EFA": 0.10,
    "EEM": 0.05,
    "IEF": 0.15,
    "TLT": 0.10,
    "LQD": 0.08,
    "HYG": 0.04,
    "GLD": 0.08,
    "DBC": 0.07,
    "VNQ": 0.05,
    "SHY": 0.03
}

default_layout = dict(
    template="plotly_white",
    height=500,
    margin=dict(t=70, l=60, r=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


## 1. Load inputs

The backtest uses the monthly asset returns and final trend signal from Notebook 2, the selected monthly volatility forecasts from Notebook 3, and the daily benchmark returns from Notebook 1. Benchmark returns are compounded to monthly frequency before comparison.


In [1181]:
def read_wide_csv(path):
    return (
        pd.read_csv(path, index_col=0, parse_dates=True)
        .sort_index()
        .apply(pd.to_numeric, errors="coerce")
    )


def compound_to_monthly(daily_returns):
    return (1.0 + daily_returns).resample("ME").prod() - 1.0


monthly_returns = read_wide_csv(TREND_DIR / "monthly_returns_aligned.csv")
signal = read_wide_csv(TREND_DIR / "final_signal.csv")
vol_forecast = read_wide_csv(VOL_DIR / "final_selected_vol_model_monthly.csv")

benchmark_returns_daily = read_wide_csv(DATA_DIR / "benchmark_returns.csv")
benchmark_returns = compound_to_monthly(benchmark_returns_daily)

data_audit = pd.DataFrame({
    "Start": [
        monthly_returns.index.min(),
        benchmark_returns.index.min(),
        signal.index.min(),
        vol_forecast.index.min(),
    ],
    "End": [
        monthly_returns.index.max(),
        benchmark_returns.index.max(),
        signal.index.max(),
        vol_forecast.index.max(),
    ],
    "Rows": [len(monthly_returns), len(benchmark_returns), len(signal), len(vol_forecast)],
    "Columns": [monthly_returns.shape[1], benchmark_returns.shape[1], signal.shape[1], vol_forecast.shape[1]],
}, index=[
    "Monthly asset returns",
    "Monthly benchmark returns",
    "Final trend signal",
    "Selected volatility forecasts",
])

display(data_audit)


,Start,End,Rows,Columns
Monthly asset returns,2008-04-30,2026-05-31,218,11
Monthly benchmark returns,2007-04-30,2026-06-30,231,1
Final trend signal,2008-04-30,2026-05-31,218,11
Selected volatility forecasts,2007-04-30,2026-06-30,231,11


## 2. Backtest assumptions

The portfolio is rebalanced monthly. Month-end signals, volatility forecasts, and covariance estimates are treated as information available after that month has closed, so the resulting weights are applied with a one-month lag.

The first evaluated return month is February 2018, because January 2018 is the first month-end with an out-of-sample volatility forecast available for the following month's allocation.


In [1182]:
OOS_START = pd.Timestamp("2018-02-28")
IS_END = pd.Timestamp("2017-12-31")
CASH_ASSET = "SHY"
WEIGHT_LAG_MONTHS = 1

RP_WINDOW_MONTHS = 36
RP_MIN_OBS = 24

CORR_WINDOW_MONTHS = 36
CORR_MIN_OBS = 24

VOL_CAP = 0.10
TRANSACTION_COST_BPS = 3.0
RISK_FREE_RATE = 0.0


## 3. Align data and enforce timing

The universe is restricted to assets available in the return, signal, and volatility forecast files. SHY is treated as the defensive cash proxy and excluded from the risky sleeve.


In [1183]:
common_assets = [
    asset for asset in monthly_returns.columns
    if asset in signal.columns and asset in vol_forecast.columns
]

risk_assets = [asset for asset in common_assets if asset != CASH_ASSET]

monthly_returns = monthly_returns.reindex(columns=common_assets).sort_index()
signal = signal.reindex(columns=common_assets).sort_index()
vol_forecast = vol_forecast.reindex(columns=common_assets).sort_index()

final_sig = signal.reindex(monthly_returns.index).ffill()
final_vol = vol_forecast.reindex(monthly_returns.index).ffill()

signal_IS = final_sig.loc[:IS_END, risk_assets]
vol_IS = final_vol.loc[:IS_END, risk_assets]

signal_OOS = final_sig.loc[OOS_START:, risk_assets]
vol_OOS = final_vol.loc[OOS_START:, risk_assets]


In [1184]:
signal_check = final_sig.loc[:IS_END, common_assets].describe().T[["min", "mean", "max"]]
signal_check

,min,mean,max
SPY,0.0000,0.7778,1.0000
EFA,0.0000,0.6047,1.0000
EEM,0.0000,0.5769,1.0000
IEF,0.0000,0.7201,1.0000
TLT,0.0000,0.6581,1.0000
LQD,0.0000,0.7564,1.0000
HYG,0.0000,0.7628,1.0000
GLD,0.0000,0.5641,1.0000
DBC,0.0000,0.4359,1.0000
VNQ,0.0000,0.7415,1.0000


In [1185]:
vol_check = final_vol.loc[:IS_END, common_assets].describe().T[["min", "mean", "max"]]
vol_check

,min,mean,max
SPY,0.0666,0.1673,0.8968
EFA,0.0679,0.2052,1.0529
EEM,0.1107,0.2584,1.5649
IEF,0.0402,0.0668,0.1311
TLT,0.0937,0.1448,0.2669
LQD,0.0402,0.0652,0.4428
HYG,0.0275,0.0931,0.6376
GLD,0.1032,0.1784,0.4436
DBC,0.0790,0.1828,0.5080
VNQ,0.0845,0.2514,1.3404


## 5. Trend + inverse-volatility portfolio

Positive trend scores define the risky sleeve. Within the active sleeve, inverse forecast volatility gives larger weights to lower-risk assets. The average trend score across risky assets determines the total risk budget; the residual is allocated to SHY.


In [ ]:
def trend_invvol_weights(
    signal,
    vol,
    all_assets,
    risk_assets,
    cash_asset=None,
    base_weights=None,
    vol_power=0.5,
    rebalance_speed=0.33,
    tactical_weight=0.6,
    min_risk_exposure=0.0,
):
    scores = signal[risk_assets].clip(0.0, 1.0)

    raw = scores / vol[risk_assets].pow(vol_power)

    sleeve_weights = raw.div(raw.sum(axis=1), axis=0).fillna(0.0)

    risk_budget = scores.mean(axis=1).clip(0.0, 1.0)
    risk_budget = risk_budget.clip(lower=min_risk_exposure)

    target_weights = pd.DataFrame(0.0, index=signal.index, columns=all_assets)
    target_weights[risk_assets] = sleeve_weights.mul(risk_budget, axis=0)

    if cash_asset is not None:
        target_weights[cash_asset] = 1.0 - target_weights[risk_assets].sum(axis=1)

    if base_weights is not None:
        base = pd.Series(base_weights, index=all_assets).fillna(0.0)
        base = base / base.sum()

        target_weights = (
            tactical_weight * target_weights
            + (1.0 - tactical_weight) * base
        )

    weights = target_weights.ewm(
        alpha=rebalance_speed,
        adjust=False
    ).mean()

    if cash_asset is not None:
        weights[cash_asset] = 1.0 - weights[risk_assets].sum(axis=1)

    return weights.clip(lower=0.0)

## IS MODEL SELECTION

### Weights

In [1187]:
strategy_params = {
    "Conservative": dict(
        vol_power=0.5,
        rebalance_speed=0.20,
        tactical_weight=0.40,
        min_risk_exposure=0.40,
    ),

    "Balanced": dict(
        vol_power=0.5,
        rebalance_speed=0.33,
        tactical_weight=0.60,
        min_risk_exposure=0.50,
    ),

    "Aggressive": dict(
        vol_power=0.75,
        rebalance_speed=0.60,
        tactical_weight=0.80,
        min_risk_exposure=0.60,
    ),

    # Higher participation, less cash drag
    "Aggressive High Exposure": dict(
        vol_power=0.75,
        rebalance_speed=0.60,
        tactical_weight=0.70,
        min_risk_exposure=0.75,
    ),

    # Smoother aggressive version
    "Aggressive Smooth": dict(
        vol_power=0.75,
        rebalance_speed=0.40,
        tactical_weight=0.70,
        min_risk_exposure=0.70,
    ),

    # Less inverse-vol concentration, more trend-driven
    "Trend Heavy": dict(
        vol_power=0.25,
        rebalance_speed=0.50,
        tactical_weight=0.75,
        min_risk_exposure=0.70,
    )
}

In [1188]:
weight_set_IS = {
    name: trend_invvol_weights(
        signal=signal_IS,
        vol=vol_IS,
        all_assets=common_assets,
        risk_assets=risk_assets,
        cash_asset=CASH_ASSET,
        base_weights=BALANCED_BENCHMARK_WEIGHTS,
        **params,
    )
    for name, params in strategy_params.items()
}

## 7. Compute portfolio returns

Raw month-end weights are lagged by one month before returns are applied. Net returns deduct transaction costs based on one-way turnover. The initial allocation is treated as portfolio funding rather than recurring turnover.


In [1189]:
def lag_weights_to_returns(raw_weights, returns_index, lag_months=1):
    return raw_weights.reindex(returns_index).ffill().shift(lag_months)


def calculate_net_returns(asset_returns, weights, tc_bps=0.0):
    index = weights.dropna(how="all").index.intersection(asset_returns.index)

    weights = weights.loc[index].fillna(0.0)
    asset_returns = asset_returns.loc[index, weights.columns]

    if asset_returns.isna().any().any():
        raise ValueError("Missing asset returns detected.")

    tc_rate = tc_bps / 10_000
    previous_weights = weights.iloc[0]

    results = []

    for i, date in enumerate(index):
        weight = weights.loc[date]
        returns = asset_returns.loc[date]

        gross_return = (weight * returns).sum()
        turnover = 0.0 if i == 0 else 0.5 * (weight - previous_weights).abs().sum()
        cost = turnover * tc_rate
        net_return = gross_return - cost

        previous_weights = weight * (1 + returns) / (1 + gross_return)

        results.append((net_return, gross_return, turnover, cost))

    results = pd.DataFrame(
        results,
        index=index,
        columns=["Net_Return", "Gross_Return", "Turnover", "Transaction_Cost"]
    )

    return (
        results["Net_Return"],
        results["Gross_Return"],
        results["Turnover"],
        results["Transaction_Cost"],
    )

In [1190]:
def strategy_output(
    raw_weight_sets,
    asset_returns,
    benchmark_returns=None,
    weight_lag_months=1,
    tc_bps=0.0,
):
    trade_weight_sets = {
        name: lag_weights_to_returns(weights, asset_returns.index, weight_lag_months)
        for name, weights in raw_weight_sets.items()
    }

    results = {
        name: calculate_net_returns(
            asset_returns=asset_returns,
            weights=weights,
            tc_bps=tc_bps,
        )
        for name, weights in trade_weight_sets.items()
    }

    strategy_returns = pd.concat(
        {name: result[0] for name, result in results.items()},
        axis=1
    ).dropna(how="any")

    turnover_table = pd.concat(
        {name: result[2] for name, result in results.items()},
        axis=1
    ).reindex(strategy_returns.index)

    transaction_cost_table = pd.concat(
        {name: result[3] for name, result in results.items()},
        axis=1
    ).reindex(strategy_returns.index)

    trade_weight_sets = {
        name: weights.reindex(strategy_returns.index).fillna(0.0)
        for name, weights in trade_weight_sets.items()
    }

    if benchmark_returns is not None:
        benchmark = benchmark_returns.reindex(strategy_returns.index).add_prefix("Benchmark: ")
        all_return_series = pd.concat([strategy_returns, benchmark], axis=1)
    else:
        benchmark = None
        all_return_series = strategy_returns.copy()

    return {
        "trade_weight_sets": trade_weight_sets,
        "strategy_returns": strategy_returns,
        "turnover_table": turnover_table,
        "transaction_cost_table": transaction_cost_table,
        "benchmark": benchmark,
        "all_return_series": all_return_series,
    }

In [1191]:
# 3) Run everything

backtest_results = strategy_output(
    raw_weight_sets=weight_set_IS,
    asset_returns=monthly_returns[common_assets].loc[:IS_END],
    benchmark_returns=benchmark_returns,
    weight_lag_months=WEIGHT_LAG_MONTHS,
    tc_bps=TRANSACTION_COST_BPS,
)

trade_weight_sets = backtest_results["trade_weight_sets"]
strategy_returns = backtest_results["strategy_returns"]
turnover_table = backtest_results["turnover_table"]
transaction_cost_table = backtest_results["transaction_cost_table"]
benchmark = backtest_results["benchmark"]
all_return_series = backtest_results["all_return_series"]

#display(strategy_returns.head())
display(strategy_returns.tail())
display(turnover_table.head())
display(transaction_cost_table.head())
display(all_return_series.head())

,Conservative,Balanced,Aggressive,Aggressive High Exposure,Aggressive Smooth,Trend Heavy
Date,,,,,,
2017-08-31,0.0103,0.0096,0.0091,0.0096,0.0096,0.0089
2017-09-30,0.0015,0.0015,0.0008,0.0010,0.0013,0.0011
2017-10-31,0.0101,0.0099,0.0102,0.0103,0.0103,0.0094
2017-11-30,0.0086,0.0079,0.0075,0.0079,0.0078,0.0073
2017-12-31,0.0131,0.0136,0.0150,0.0147,0.0144,0.0135


,Conservative,Balanced,Aggressive,Aggressive High Exposure,Aggressive Smooth,Trend Heavy
Date,,,,,,
2008-05-31,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2008-06-30,0.0101,0.0166,0.0382,0.0205,0.0138,0.0184
2008-07-31,0.0287,0.0361,0.0649,0.0648,0.0342,0.0581
2008-08-31,0.0225,0.0331,0.0728,0.0817,0.0523,0.0612
2008-09-30,0.0199,0.0259,0.0423,0.0459,0.0405,0.0480


,Conservative,Balanced,Aggressive,Aggressive High Exposure,Aggressive Smooth,Trend Heavy
Date,,,,,,
2008-05-31,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2008-06-30,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2008-07-31,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2008-08-31,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2008-09-30,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


,Conservative,Balanced,Aggressive,Aggressive High Exposure,Aggressive Smooth,Trend Heavy,Benchmark: Balanced Benchmark Portfolio
Date,,,,,,,
2008-05-31,0.0058,0.0064,0.0085,0.0091,0.0084,0.0056,0.0051
2008-06-30,-0.0168,-0.0119,-0.0062,-0.0091,-0.0090,-0.0093,-0.0275
2008-07-31,-0.0144,-0.0152,-0.0197,-0.0228,-0.0201,-0.0178,-0.0121
2008-08-31,-0.0124,-0.0144,-0.0195,-0.0226,-0.0209,-0.0181,-0.0090
2008-09-30,-0.0436,-0.0342,-0.0244,-0.0325,-0.0341,-0.0288,-0.0526


## 8. performance diagnostics

Performance is evaluated on the common OOS sample after the trading lag is applied. The main question is whether the adaptive portfolio delivers enough drawdown reduction and risk-adjusted improvement to justify lower upside capture versus Risk Parity.


In [1192]:
def drawdown_series(returns):
    wealth = (1.0 + returns.dropna()).cumprod()
    return wealth / wealth.cummax() - 1.0


def performance_summary(
    return_table,
    benchmark_col=None,
    periods_per_year=12,
    risk_free_rate=0.0,
):
    rows = {}

    for name in return_table.columns:
        returns = return_table[name].dropna()

        if returns.empty:
            continue

        ann_return = (1.0 + returns).prod() ** (periods_per_year / len(returns)) - 1.0
        ann_vol = returns.std(ddof=1) * np.sqrt(periods_per_year)

        downside = returns[returns < 0]
        downside_vol = (
            downside.std(ddof=1) * np.sqrt(periods_per_year)
            if len(downside) > 1
            else np.nan
        )

        mdd = drawdown_series(returns).min()

        worst_3m = (
            (1.0 + returns)
            .rolling(3)
            .apply(np.prod, raw=True)
            .sub(1.0)
            .min()
        )

        rows[name] = {
            "Annual Return": ann_return,
            "Annual Volatility": ann_vol,
            "Sharpe": (ann_return - risk_free_rate) / ann_vol if ann_vol > 0 else np.nan,
            "Sortino": (ann_return - risk_free_rate) / downside_vol if downside_vol > 0 else np.nan,
            "Max Drawdown": mdd,
            "Calmar": ann_return / abs(mdd) if mdd < 0 else np.nan,
            "Worst 3M": worst_3m,
        }

    summary = pd.DataFrame(rows).T

    if benchmark_col is not None and benchmark_col in summary.index:
        benchmark = summary.loc[benchmark_col]

        summary["Excess Annual Return"] = (
            summary["Annual Return"] - benchmark["Annual Return"]
        )

        summary["Excess Sharpe"] = (
            summary["Sharpe"] - benchmark["Sharpe"]
        )

        summary["Excess Max Drawdown"] = (
            summary["Max Drawdown"] - benchmark["Max Drawdown"]
        )

    return summary

In [1193]:
# Use the combined strategy + benchmark table from backtest_results

summary_table = performance_summary(
    return_table=backtest_results["all_return_series"],
    benchmark_col=backtest_results["benchmark"].columns[0],
    periods_per_year=12,
    risk_free_rate=0.0,
)

display(summary_table.style.format({
    "Annual Return": "{:.2%}",
    "Annual Volatility": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}",
    "Max Drawdown": "{:.2%}",
    "Calmar": "{:.2f}",
    "Worst 3M": "{:.2%}",
    "Excess Annual Return": "{:.2%}",
    "Excess Sharpe": "{:.2f}",
    "Excess Max Drawdown": "{:.2%}",
}))

,Annual Return,Annual Volatility,Sharpe,Sortino,Max Drawdown,Calmar,Worst 3M,Excess Annual Return,Excess Sharpe,Excess Max Drawdown
Conservative,4.81%,8.18%,0.59,0.71,-22.30%,0.22,-15.87%,-1.18%,-0.05,4.11%
Balanced,4.58%,7.66%,0.60,0.73,-18.40%,0.25,-14.36%,-1.41%,-0.04,8.01%
Aggressive,4.39%,7.83%,0.56,0.68,-17.02%,0.26,-14.83%,-1.60%,-0.08,9.39%
Aggressive High Exposure,4.61%,8.69%,0.53,0.62,-20.56%,0.22,-17.95%,-1.38%,-0.11,5.85%
Aggressive Smooth,4.40%,8.42%,0.52,0.61,-19.91%,0.22,-17.12%,-1.59%,-0.11,6.50%
Trend Heavy,4.64%,7.70%,0.60,0.73,-16.99%,0.27,-14.69%,-1.35%,-0.04,9.42%
Benchmark: Balanced Benchmark Portfolio,5.99%,9.40%,0.64,0.76,-26.41%,0.23,-17.76%,0.00%,0.00,0.00%


In [1194]:
turnover_summary = pd.DataFrame({
    "Average Monthly Turnover": turnover_table.mean(),
    "Median Monthly Turnover": turnover_table.median(),
    "Annualized TC Drag": transaction_cost_table.mean() * 12,
    "Max Monthly Turnover": turnover_table.max(),
})

turnover_summary.style.format({
    "Average Monthly Turnover": "{:.2%}",
    "Median Monthly Turnover": "{:.2%}",
    "Annualized TC Drag": "{:.2%}",
    "Max Monthly Turnover": "{:.2%}",
})


,Average Monthly Turnover,Median Monthly Turnover,Annualized TC Drag,Max Monthly Turnover
Conservative,1.62%,1.58%,0.01%,4.64%
Balanced,2.94%,2.78%,0.01%,6.90%
Aggressive,6.71%,5.89%,0.02%,19.04%
Aggressive High Exposure,5.87%,5.12%,0.02%,19.93%
Aggressive Smooth,4.08%,3.58%,0.01%,12.24%
Trend Heavy,4.79%,4.35%,0.02%,14.33%


In [1195]:
wealth = (1.0 + all_return_series.dropna(how="all")).cumprod()

fig = go.Figure()
for name in wealth.columns:
    fig.add_trace(go.Scatter(x=wealth.index, y=wealth[name], mode="lines", name=name))

fig.update_layout(
    **default_layout,
    title="Cumulative Growth of $1",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
)
fig.show()

In [1196]:
drawdowns = all_return_series.apply(drawdown_series)

fig = go.Figure()
for name in drawdowns.columns:
    fig.add_trace(go.Scatter(x=drawdowns.index, y=drawdowns[name], mode="lines", name=name))

fig.update_layout(
    **default_layout,
    title="OOS Drawdowns",
    xaxis_title="Date",
    yaxis_title="Drawdown",
)
fig.update_yaxes(tickformat=".0%")
fig.show()


In [1197]:
for name, weights in trade_weight_sets.items():
    fig = go.Figure()

    for asset in weights.columns:
        fig.add_trace(
            go.Scatter(
                x=weights.index,
                y=weights[asset],
                mode="lines",
                name=asset,
                stackgroup="one",
            )
        )

    fig.update_layout(
        **default_layout,
        title=f"{name} — Portfolio Weights",
        xaxis_title="Date",
        yaxis_title="Weight",
    )
    fig.update_yaxes(tickformat=".0%")
    fig.show()


## OOS

In [1198]:
weights_OOS = {
    "Aggressive OOS": trend_invvol_weights(
        signal=signal_OOS,
        vol=vol_OOS,
        all_assets=common_assets,
        risk_assets=risk_assets,
        cash_asset=CASH_ASSET,
        base_weights=BALANCED_BENCHMARK_WEIGHTS,
        **strategy_params["Aggressive"],
    )
}

In [1199]:
# 3) Run everything

OOS_results = strategy_output(
    raw_weight_sets=weights_OOS,
    asset_returns=monthly_returns[common_assets].loc[OOS_START:],
    benchmark_returns=benchmark_returns,
    weight_lag_months=WEIGHT_LAG_MONTHS,
    tc_bps=TRANSACTION_COST_BPS,
)

trade_weight_sets = OOS_results["trade_weight_sets"]
strategy_returns = OOS_results["strategy_returns"]
turnover_table = OOS_results["turnover_table"]
transaction_cost_table = OOS_results["transaction_cost_table"]
benchmark = OOS_results["benchmark"]
all_return_series = OOS_results["all_return_series"]

display(strategy_returns.head())
display(strategy_returns.tail())
display(turnover_table.head())
display(transaction_cost_table.head())
display(all_return_series.head())

,Aggressive OOS
Date,
2018-03-31,0.0006
2018-04-30,-0.0016
2018-05-31,0.0026
2018-06-30,-0.0080
2018-07-31,0.0066


,Aggressive OOS
Date,
2026-01-31,0.0471
2026-02-28,0.0391
2026-03-31,-0.0387
2026-04-30,0.0450
2026-05-31,0.0101


,Aggressive OOS
Date,
2018-03-31,0.0000
2018-04-30,0.0190
2018-05-31,0.0292
2018-06-30,0.0490
2018-07-31,0.0696


,Aggressive OOS
Date,
2018-03-31,0.0000
2018-04-30,0.0000
2018-05-31,0.0000
2018-06-30,0.0000
2018-07-31,0.0000


,Aggressive OOS,Benchmark: Balanced Benchmark Portfolio
Date,,
2018-03-31,0.0006,0.0020
2018-04-30,-0.0016,-0.0015
2018-05-31,0.0026,0.0098
2018-06-30,-0.0080,-0.0039
2018-07-31,0.0066,0.0102


In [1200]:
# Use the combined strategy + benchmark table from backtest_results

summary_table = performance_summary(
    return_table=OOS_results["all_return_series"],
    benchmark_col=OOS_results["benchmark"].columns[0],
    periods_per_year=12,
    risk_free_rate=0.0,
)

display(summary_table.style.format({
    "Annual Return": "{:.2%}",
    "Annual Volatility": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}",
    "Max Drawdown": "{:.2%}",
    "Calmar": "{:.2f}",
    "Worst 3M": "{:.2%}",
    "Excess Annual Return": "{:.2%}",
    "Excess Sharpe": "{:.2f}",
    "Excess Max Drawdown": "{:.2%}",
}))

,Annual Return,Annual Volatility,Sharpe,Sortino,Max Drawdown,Calmar,Worst 3M,Excess Annual Return,Excess Sharpe,Excess Max Drawdown
Aggressive OOS,6.97%,7.70%,0.90,1.34,-12.90%,0.54,-8.89%,-1.49%,0.03,6.16%
Benchmark: Balanced Benchmark Portfolio,8.46%,9.67%,0.87,1.36,-19.07%,0.44,-9.91%,0.00%,0.00,0.00%


In [1201]:
def allocation_summary(weights_dict, CASH_ASSET=None):
    rows = {}

    for name, weights in weights_dict.items():
        risk_cols = [col for col in weights.columns if col != CASH_ASSET]
        cash_weight = weights[CASH_ASSET] if CASH_ASSET in weights.columns else pd.Series(0.0, index=weights.index)

        rows[name] = {
            "Average Gross Exposure": weights.abs().sum(axis=1).mean(),
            "Max Gross Exposure": weights.abs().sum(axis=1).max(),
            "Average Cash Weight": cash_weight.mean(),
            "Max Cash Weight": cash_weight.max(),
            "Average Active Risk Assets": (weights[risk_cols] > 0.01).sum(axis=1).mean(),
            "Max Single Asset Weight": weights[risk_cols].max(axis=1).max(),
        }

    return pd.DataFrame(rows).T

allocation_table = allocation_summary(trade_weight_sets, CASH_ASSET=CASH_ASSET)
allocation_table.style.format({
    "Average Gross Exposure": "{:.2%}",
    "Max Gross Exposure": "{:.2%}",
    "Average Cash Weight": "{:.2%}",
    "Max Cash Weight": "{:.2%}",
    "Average Active Risk Assets": "{:.1f}",
    "Max Single Asset Weight": "{:.2%}",
})


,Average Gross Exposure,Max Gross Exposure,Average Cash Weight,Max Cash Weight,Average Active Risk Assets,Max Single Asset Weight
Aggressive OOS,100.00%,100.00%,23.02%,32.60%,9.9,48.20%


In [1202]:
name = "Aggressive OOS"

fig = go.Figure()

for asset in weights_OOS[name].columns:
    fig.add_trace(
        go.Scatter(
            x=weights_OOS[name].index,
            y=weights_OOS[name][asset],
            mode="lines",
            name=asset,
            stackgroup="one",
        )
    )

fig.update_layout(
    **default_layout,
    title=f"{name} — Portfolio Weights",
    xaxis_title="Date",
    yaxis_title="Weight",
)

fig.update_yaxes(tickformat=".0%")
fig.show()

In [1203]:
def capture_summary(strategy_returns, benchmark_returns):
    benchmark = benchmark_returns.squeeze().reindex(strategy_returns.index)

    rows = {}

    for name in strategy_returns.columns:
        r = strategy_returns[name].dropna()
        b = benchmark.reindex(r.index)

        up = b > 0
        down = b < 0

        rows[name] = {
            "Upside Capture": r[up].mean() / b[up].mean(),
            "Downside Capture": r[down].mean() / b[down].mean(),
            "Positive Months": (r > 0).mean(),
            "Benchmark Positive Months": (b > 0).mean(),
        }

    return pd.DataFrame(rows).T

In [1204]:
capture_table = capture_summary(
    strategy_returns=OOS_results["strategy_returns"],
    benchmark_returns=benchmark_returns,
)

display(capture_table.style.format({
    "Upside Capture": "{:.2%}",
    "Downside Capture": "{:.2%}",
    "Positive Months": "{:.2%}",
    "Benchmark Positive Months": "{:.2%}",
}))

,Upside Capture,Downside Capture,Positive Months,Benchmark Positive Months
Aggressive OOS,79.97%,78.27%,68.69%,68.69%


## 10. Calendar-year returns

Calendar-year returns provide a final stability check and make it easier to identify which market environments drove the strategy comparison.


In [1205]:
calendar_returns = all_return_series.groupby(all_return_series.index.year).apply(
    lambda x: (1.0 + x).prod() - 1.0
)

calendar_returns.style.format("{:.2%}")


,Aggressive OOS,Benchmark: Balanced Benchmark Portfolio
Date,,
2018,-4.85%,-3.26%
2019,15.72%,19.49%
2020,5.93%,13.85%
2021,10.70%,11.10%
2022,-10.00%,-14.22%
2023,7.75%,12.32%
2024,7.19%,8.85%
2025,17.63%,17.66%
2026,10.40%,8.26%


In [1206]:
strategy_returns.to_csv(OUT_DIR / "strategy_returns_net.csv")
turnover_table.to_csv(OUT_DIR / "turnover.csv")
summary_table.to_csv(OUT_DIR / "performance_summary.csv")
allocation_table.to_csv(OUT_DIR / "allocation_summary.csv")

print("Saved backtest outputs to:", OUT_DIR.resolve())


Saved backtest outputs to: C:\Users\mmodr\Desktop\project-trend-invvol-strategy\results\backtest_model_portfolio
